In [2]:
import cv2 
from matplotlib import pyplot as plt
from PIL import Image
import pickle
import shutil
import os
import glob
import csv
import numpy as np
from collections import OrderedDict


In [35]:
pickle_file = open('/media/osero/SamsungSSD/pickles/features_right_hand_frames_small.pickle', 'rb')
features,labels = pickle.load(pickle_file)

In [36]:
# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256" 
video_labels = []  # Populate this with the corresponding labels for each video
process_count = 0

folder_path_list = []
sorted_features = []
for label_folder in os.listdir(video_folder):
    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    # print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in os.listdir(full_label_folder):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        folder_path_list.append(full_sample_folder)
        feature_path_list = []
        for image_file in os.listdir(full_sample_folder):
            full_image_file = os.path.join(full_sample_folder, image_file)
            feature_path_list.append(full_image_file)
        # image = Image.open(full_image_file)
        #     image_list.append(image)
        # video_embedding = extract_video_embedding(image_list)
        # video_embeddings.append(video_embedding)
        # video_labels.append(label)

        feature_dict=dict(zip(feature_path_list, features[process_count]))
        ordered_feature_dict = OrderedDict(sorted(feature_dict.items()))
        ordered_feature_single_image = list(ordered_feature_dict.values())
        sorted_features.append(ordered_feature_single_image)
        process_count += 1
            
ccc = 4
        

In [37]:
zipped_pickle_feature_dict=dict(zip(folder_path_list, sorted_features))
ordered_pickle_feature_dict= OrderedDict(sorted(zipped_pickle_feature_dict.items()))
zipped_pickle_labels_dict=dict(zip(folder_path_list, labels))
ordered_pickle_labels_dict= OrderedDict(sorted(zipped_pickle_labels_dict.items()))

ordered_features = list(ordered_pickle_feature_dict.values())
ordered_labels = list(ordered_pickle_labels_dict.values())
ordered_paths = list(ordered_pickle_feature_dict.keys())

ccc = 4

In [38]:

with open('/media/osero/SamsungSSD/pickles/features_right_hand_frames_small_ordered.pickle', 'wb') as handle:
    pickle.dump((ordered_paths, ordered_features, ordered_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [39]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Step 3: Function to Extract Embeddings for a List of Frames
def extract_video_embedding(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        for image in image_list:
            input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
            features = model(input_tensor)
            embeddings.append(features.squeeze().cpu().numpy())
    return embeddings

# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256" 
video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
process_count = 0
for label_folder in sorted(os.listdir(video_folder)):
    process_count += 1
    if (process_count > 3):
        break
    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in sorted(os.listdir(full_label_folder)):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        print("full_sample_folder: ", full_sample_folder)
        image_list = []
        for image_file in sorted(os.listdir(full_sample_folder)):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            image_list.append(image)
            
        video_embedding = extract_video_embedding(image_list)
        video_embeddings.append(video_embedding)
        if (video_embedding[5] != ordered_features[len(video_embeddings)-1]).all():
            print('@@@@@@@@@@@@@@@@@@@')
        video_labels.append(label)

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main


process_count:  1  , label:  1
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_001
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_002
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_003
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_004
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_001
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_002
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_003
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_004
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_005
full_sample_folder:  /media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_006
full_sample_folder:  /media/osero/SamsungSSD/CMPE

In [42]:
pickle_file1 = open('/media/osero/SamsungSSD/pickles/features_right_hand_frames_small_ordered.pickle', 'rb')
paths1, features1,labels1 = pickle.load(pickle_file1)
print(paths1[0:10])
print(labels1[0:10])
print(paths1[-10:])
print(labels1[-10:])

['/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_001', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_002', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_003', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_2_004', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_001', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_002', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_003', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_004', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_005', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0001/User_4_006']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
['/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0744/User_6_001', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0744/User_6_002', '/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256/0744/User_6_003', '/med

In [10]:

def get_active_frames(input_raw) -> np.ndarray:
    threshold = (
        (((input_raw["pose"]["left_hip"][:, 1] + input_raw["pose"]["right_hip"][:, 1]) / 2 )* 7)
        + input_raw["pose"]["nose"][:, 1]
    ) / 10

    active_frames = (
        np.minimum(
            input_raw["hand_left"]["left_lunate_bone"][:, 1],
            input_raw["hand_right"]["right_lunate_bone"][:, 1],
        )
        < threshold
    )

    active_frame_indices = np.argwhere(active_frames).squeeze()
    return active_frame_indices

In [13]:
file = open('/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/0001/User_2_001.pickle', 'rb')
input_raw = pickle.load(file)

active_frame_indices = get_active_frames(input_raw)
input_raw = {
    **input_raw["pose"],
    **input_raw["face"],
    **input_raw["hand_left"],
    **input_raw["hand_right"],
}

print('active_frame_indices: ', active_frame_indices)
print('Len check: ', (len(active_frame_indices)-1) == (max(active_frame_indices) - min(active_frame_indices)))
ccc = 3
# input = np.array([input_raw[jn] for jn in nodes]).transpose((1, 0, 2))
# active_frame_indices = (
#     active_frame_indices
#     if active_frame_indices.size > 10
#     else np.arange(0, len(input))
# )

# input = input[active_frame_indices, ...]

active_frame_indices:  [ 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32
 33 34 35 36 37 38 39 40 41 42 43 44]
Len check:  True
